# LSTM (Long Short_Term Memory)
- RNN의 기울기 소실 문제를 해결하기 위해 고안한 모델입니다
- RNN은 은닉 상태 하나로 모든 걸 처리하지만 LSTM은 셀 상태라는 별도의 통로가 추가됩니다
- 이 통로에 무엇을 넣고 뺄지는 게이트가 조절합니다.

## 셀 상태
- RNN에서는 은닉 상태가 매시점마다 tanh를 통과해서 다음 시점에 전달되지만 LSTM은 셀 상태에 게이트값을 곱하고 더해 다음 시점으로 넘깁니다
- 셀 상태는 다음 시점으로 넘어갈 때 tanh를 적용하지 않아서 값이 -1 ~ 1로 압축되지 않습니다
- 이를 통해 기울기 소실 문제를 해결할 수 있습니다.

## 게이트
- 게이트는 각 원소들이 0 ~ 1 사이의 값을 가지는 벡터입니다.
- 게이트들은 각각 기존 것을 얼마나 잊고, 새로 쓰고, 내보낼지 결정합니다.
- 각 원소의 값들이 0 ~ 1 범위의 실수값을 가지게 하기위해 시그모이드를 통과시킵니다.

1. forget gate(f_t)
    - 이전 시점 셀 상태에서 각 원소를 얼마나 잊을지 결정하는 게이트입니다
2. input gate(i_t)
    - 이번 입력에서 새로 만들어낸 정보 (g_t)를 얼마나 셀 상태에 반영할지 결정하는 게이트입니다.
3. output gate(o_t)
    - 이번 시점에 갱신된 셀 상태를 외부에 얼마나 노출시킬지 결정합니다

- g_t : RNN으로 치면 이번 시점에 갱신된 은닉 상태의 값

## 수식

- 현재 시점의 셀 상태 c_t는 다음과 같이 정의됩니다.
    - c_t = f_t $\odot$ c_{t-1} + i_t $\odot$ g_t
- g_t = tanh(W_g $\cdot$ [h_{t-1},x_t] + b_g)
- 각 게이트 : $\sigma$(W $\cdot$ [h_{t-1},x_t] + b)
- 은닉 상태 : o_t $\odot$ tanh(c_t)

In [7]:
# LSTM의 기본 구현 저번주 RNN과 동일한 예제로 결과 확인
import numpy as np

GATES = ["f", "i", "g", "o"]


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


class LSTM:

    def __init__(self, gates, Why, b_y):
        self.gates = {name: list(gates[name]) for name in GATES}
        self.Why = Why
        self.b_y = b_y

        self.grads = {name: [np.zeros_like(p) for p in self.gates[name]] for name in GATES}
        self.grads_sum = {name: [np.zeros_like(p) for p in self.gates[name]] for name in GATES}
        self.dWhy_sum = np.zeros_like(Why)
        self.db_y_sum = 0.0

        self.cache = None

    def forward(self, x, h_prev, c_prev):
        acts = {}
        for name in GATES:
            Wx, Wh, b = self.gates[name]
            z = Wx @ x + Wh @ h_prev + b
            acts[name] = np.tanh(z) if name == "g" else sigmoid(z)
        f, i, g, o = acts["f"], acts["i"], acts["g"], acts["o"]

        c_next = f * c_prev + i * g
        h_next = o * np.tanh(c_next)
        y = self.Why @ h_next + self.b_y

        self.cache = (x, h_prev, c_prev, f, i, g, o, c_next, h_next)
        return h_next, c_next, y

    def error(self, y, target):
        return np.round((target - y) ** 2, 4)

    def backward(self, y, target, dh_next, dc_next):
        x, h_prev, c_prev, f, i, g, o, c_next, h_next = self.cache

        dy = 2 * (y - target)
        dh = dy * self.Why + dh_next

        tanh_c = np.tanh(c_next)
        do = dh * tanh_c
        dc = dh * o * (1 - tanh_c ** 2) + dc_next

        raw = {
            "f": dc * c_prev * f * (1 - f),          
            "i": dc * g * i * (1 - i),               
            "g": dc * i * (1 - g ** 2),               
            "o": do * o * (1 - o),                    
        }

        dh_prev = np.zeros_like(h_prev, dtype=float)
        for name in GATES:
            Wx, Wh, b = self.gates[name]
            r = raw[name]
            dWx, dWh, db = np.outer(r, x), np.outer(r, h_prev), r
            self.grads[name] = [dWx, dWh, db]
            self.grads_sum[name][0] += dWx
            self.grads_sum[name][1] += dWh
            self.grads_sum[name][2] += db
            dh_prev = dh_prev + Wh.T @ r

        dc_prev = dc * f

        dWhy = dy * h_next
        db_y = dy
        self.dWhy_sum += dWhy
        self.db_y_sum += db_y

        return dh_prev, dc_prev

    def update(self, lr):
        for name in GATES:
            for k in range(3):
                self.gates[name][k] -= lr * self.grads_sum[name][k]
        self.Why -= lr * self.dWhy_sum
        self.b_y -= lr * self.db_y_sum

    def reset_grads_sum(self):
        for name in GATES:
            for g in self.grads_sum[name]:
                g[...] = 0
        self.dWhy_sum[...] = 0
        self.db_y_sum = 0.0

# 지난주와 동일 예제
Wx0 = np.array([[0.1, 0.2], [0.3, 0.4]])
Wh0 = np.array([[0.5, 0.0], [0.0, 0.5]])
b0 = np.array([0.1, 0.1])
Why = np.array([0.5, 0.5])
b_y = 0.1

gates = {name: [Wx0.copy(), Wh0.copy(), b0.copy()] for name in GATES}
lstm = LSTM(gates, Why.copy(), b_y)

xs = [np.array([1.0, 2.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0])]
target = 0.0
lr = 0.05
rep = 3

for epoch in range(1, rep + 1):
    h = np.zeros(2)
    c = np.zeros(2)
    caches = []
    ys = []
    i = 0
    for x in xs:
 
        h, c, y = lstm.forward(x, h, c)
        caches.append(lstm.cache)
        ys.append(y)
        print(f"t_{i + 1} \tgates(f,i,o): {np.round(caches[i][3],4)}{np.round(caches[i][4],4)}{np.round(caches[i][6],4)}\n")
        i+=1

    total_loss = sum(lstm.error(np.array(ys),target))
    print(f"epoch {epoch}  ys={[round(float(y), 4) for y in ys]}  loss={total_loss:.4f}")
    

    if epoch == rep:
        break

    lstm.reset_grads_sum()
    dh_next = np.zeros(2)
    dc_next = np.zeros(2)
    for t in reversed(range(len(xs))):
        lstm.cache = caches[t]
        dh_next, dc_next = lstm.backward(ys[t], target, dh_next, dc_next)

    lstm.update(lr)


t_1 	gates(f,i,o): [0.6457 0.7685][0.6457 0.7685][0.6457 0.7685]

t_2 	gates(f,i,o): [0.6005 0.672 ][0.6005 0.672 ][0.6005 0.672 ]

t_3 	gates(f,i,o): [0.6281 0.7371][0.6281 0.7371][0.6281 0.7371]

epoch 1  ys=[0.4249, 0.4554, 0.5703]  loss=0.7132
t_1 	gates(f,i,o): [0.6431 0.7667][0.6376 0.7629][0.6373 0.7599]

t_2 	gates(f,i,o): [0.5932 0.6688][0.5902 0.6665][0.5898 0.6642]

t_3 	gates(f,i,o): [0.6193 0.7338][0.6154 0.731 ][0.6149 0.7284]

epoch 2  ys=[0.2112, 0.2313, 0.324]  loss=0.2031
t_1 	gates(f,i,o): [0.642  0.7658][0.6342 0.7603][0.6339 0.756 ]

t_2 	gates(f,i,o): [0.589  0.6673][0.5848 0.6639][0.5843 0.6605]

t_3 	gates(f,i,o): [0.6141 0.7321][0.6087 0.7281][0.6081 0.7243]

epoch 3  ys=[0.1, 0.1149, 0.1955]  loss=0.0614


## RNN과의 비교 1: 이론적으로 왜 LSTM이 나은가
- RNN의 dh_prev = Wh^T $\cdot$ dt 에는 매 시점마다 tanh 미분과 Wh가 반복해서 곱해집니다. 시퀀스가 길어질수록 이 반복곱 때문에 기울기가 지수적으로 작아지는(또는 커지는) 문제가 생깁니다.
- LSTM의 dc_prev = dc_t $\odot$ f_t 에는 tanh 미분이 곱해지지 않고 f_t(forget gate, 0~1)만 곱해집니다. f_t가 1에 가깝게 학습되면 셀 상태를 통한 기울기가 거의 그대로 보존되므로, 이론적으로는 시퀀스가 길수록 LSTM이 RNN보다 유리해야 합니다.
- 이론적 예측: 시퀀스가 짧으면 두 모델 차이가 작고, 시퀀스가 길어질수록 LSTM이 RNN보다 확실히 나아야 합니다.

## RNN과의 비교 2: 실험 설계
- 위에서 구현한 예제는 저번주와 같은 데이터에서 동작하게 구현했지만 시퀀스의 크기도 작고 예제의 숫자가 간단하고 데이터의 크기도 크지 않습니다.
- 손계산을 위함이었지만 이정도 결과로는 LSTM과 RNN을 비교했을때 유의미한 결과가 없을거라 판단했습니다.
- 더 다양한 비교를 위해 교수님 피드백을 반영해서 아래와 같이 설계했습니다.

1. 장기 의존성 태스크(copy task): 시퀀스의 첫 입력값 하나만 기억했다가 마지막 시점에 그대로 출력. 그 사이 입력은 전부 0(정보 없음). 시퀀스 전체를 기억해야 풀리는, 이론과 직결되는 태스크입니다. 지난주에 작성했던 기울기 소실 문제를 보이기 위한 예제와 비슷하게 설계했습니다.
2. 단기 의존성 태스크: 매 시점 target이 바로 직전 입력에만 의존합니다. 먼 과거를 기억할 필요가 없습니다.
3. 시퀀스 길이(T)를 변화시켜서 이론 예측(길수록 LSTM이 유리)이 실제로 나타나는지 확인합니다.
4. 서로 다른 랜덤 시드로 3회씩 반복해서 평균과 표준편차를 같이 봅니다. 한 번 우연히 잘 나온 것과 일관되게 나은 것을 구분하기 위함입니다.
5. 파라미터 수는 작게(H=4), 데이터도 적게(샘플 10개) 유지해서 계산이 오래 걸리지 않게 했습니다.

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


class SimpleRNN:
    def __init__(self, D, H, seed):
        rng = np.random.RandomState(seed)
        s = 1 / np.sqrt(D + H)
        self.Wx, self.Wh, self.bh = rng.randn(H, D) * s, rng.randn(H, H) * s, np.zeros(H)
        self.Why, self.by = rng.randn(H) * s, 0.0
        self.H = H

    def train(self, dataset, epochs, lr):
        losses = []
        for ep in range(epochs):
            dWx = np.zeros_like(self.Wx); dWh = np.zeros_like(self.Wh); dbh = np.zeros_like(self.bh)
            dWhy = np.zeros_like(self.Why); dby = 0.0; total = 0.0
            for xs, out_ts, targets in dataset:
                T = len(xs)
                h = np.zeros(self.H); hs = [h]
                for x in xs:
                    h = np.tanh(self.Wx @ x + self.Wh @ h + self.bh); hs.append(h)
                dh_next = np.zeros(self.H)
                for t in range(T, 0, -1):
                    dh = dh_next.copy()
                    if t in out_ts:
                        y = self.Why @ hs[t] + self.by; target = targets[t]
                        total += (target - y) ** 2
                        dy = 2 * (y - target); dh = dh + dy * self.Why
                        dWhy += dy * hs[t]; dby += dy
                    dt = dh * (1 - hs[t] ** 2)
                    dWx += np.outer(dt, xs[t - 1]); dWh += np.outer(dt, hs[t - 1]); dbh += dt
                    dh_next = self.Wh.T @ dt
            n = len(dataset)
            self.Wx -= lr * dWx / n; self.Wh -= lr * dWh / n; self.bh -= lr * dbh / n
            self.Why -= lr * dWhy / n; self.by -= lr * dby / n
            losses.append(total / n)
        return losses


class SimpleLSTM:
    def __init__(self, D, H, seed, forget_bias=1.0):
        rng = np.random.RandomState(seed)
        s = 1 / np.sqrt(D + H)
        W = lambda: rng.randn(H, D + H) * s
        self.Wf, self.Wi, self.Wg, self.Wo = W(), W(), W(), W()
        self.bf = np.full(H, forget_bias); self.bi = np.zeros(H); self.bg = np.zeros(H); self.bo = np.zeros(H)
        self.Why = rng.randn(H) * s; self.by = 0.0
        self.H = H

    def train(self, dataset, epochs, lr):
        H = self.H
        losses = []
        for ep in range(epochs):
            dWf = np.zeros_like(self.Wf); dWi = np.zeros_like(self.Wi)
            dWg = np.zeros_like(self.Wg); dWo = np.zeros_like(self.Wo)
            dbf = np.zeros_like(self.bf); dbi = np.zeros_like(self.bi)
            dbg = np.zeros_like(self.bg); dbo = np.zeros_like(self.bo)
            dWhy = np.zeros_like(self.Why); dby = 0.0; total = 0.0
            for xs, out_ts, targets in dataset:
                T = len(xs)
                h = np.zeros(H); c = np.zeros(H); hs = [h]; cs = [c]; cache = []
                for x in xs:
                    z = np.concatenate([h, x])
                    f = sigmoid(self.Wf @ z + self.bf); i = sigmoid(self.Wi @ z + self.bi)
                    g = np.tanh(self.Wg @ z + self.bg); o = sigmoid(self.Wo @ z + self.bo)
                    c = f * c + i * g; h = o * np.tanh(c)
                    hs.append(h); cs.append(c); cache.append((f, i, g, o, z))
                dh_next = np.zeros(H); dc_next = np.zeros(H)
                for t in range(T, 0, -1):
                    f, i, g, o, z = cache[t - 1]; c_, c_prev, h_ = cs[t], cs[t - 1], hs[t]
                    dh = dh_next.copy()
                    if t in out_ts:
                        y = self.Why @ h_ + self.by; target = targets[t]
                        total += (target - y) ** 2
                        dy = 2 * (y - target); dh = dh + dy * self.Why
                        dWhy += dy * h_; dby += dy
                    tanh_c = np.tanh(c_)
                    do = dh * tanh_c; dc = dh * o * (1 - tanh_c ** 2) + dc_next
                    df = dc * c_prev; di = dc * g; dg = dc * i
                    do_r = do * o * (1 - o); df_r = df * f * (1 - f)
                    di_r = di * i * (1 - i); dg_r = dg * (1 - g ** 2)
                    dWf += np.outer(df_r, z); dbf += df_r
                    dWi += np.outer(di_r, z); dbi += di_r
                    dWg += np.outer(dg_r, z); dbg += dg_r
                    dWo += np.outer(do_r, z); dbo += do_r
                    dz = self.Wf.T @ df_r + self.Wi.T @ di_r + self.Wg.T @ dg_r + self.Wo.T @ do_r
                    dh_next = dz[:H]; dc_next = dc * f
            n = len(dataset)
            self.Wf -= lr * dWf / n; self.Wi -= lr * dWi / n
            self.Wg -= lr * dWg / n; self.Wo -= lr * dWo / n
            self.bf -= lr * dbf / n; self.bi -= lr * dbi / n
            self.bg -= lr * dbg / n; self.bo -= lr * dbo / n
            self.Why -= lr * dWhy / n; self.by -= lr * dby / n
            losses.append(total / n)
        return losses


def make_copy_task(N, T, seed):
    """장기 의존성: x1만 신호, 나머지는 0. target은 t=T에서만 존재하고 target_T = x1"""
    rng = np.random.RandomState(seed)
    dataset = []
    for _ in range(N):
        x1 = rng.uniform(-1, 1)
        xs = [np.array([x1])] + [np.array([0.0]) for _ in range(T - 1)]
        dataset.append((xs, [T], {T: x1}))
    return dataset


def make_shortdep_task(N, T, seed):
    """단기 의존성: target_t는 바로 직전 입력에만 의존"""
    rng = np.random.RandomState(seed)
    dataset = []
    for _ in range(N):
        raw = rng.uniform(-1, 1, size=T)
        xs = [np.array([raw[t]]) for t in range(T)]
        targets, prev = {}, 0.0
        for t in range(1, T + 1):
            targets[t] = 0.5 * raw[t - 1] + 0.5 * prev
            prev = raw[t - 1]
        dataset.append((xs, list(range(1, T + 1)), targets))
    return dataset

In [9]:
H = 4
seeds = [0, 1, 2]

print('=== 1) 장기 의존성(copy task): 시퀀스 길이별 RNN vs LSTM ===')
for T in [5, 15, 30]:
    ds = make_copy_task(10, T, seed=0)
    rnn_losses, lstm_losses = [], []
    for seed in seeds:
        rnn = SimpleRNN(1, H, seed=seed)
        rnn_losses.append(min(rnn.train(ds, epochs=200, lr=0.1)))
        lstm = SimpleLSTM(1, H, seed=seed)
        lstm_losses.append(min(lstm.train(ds, epochs=200, lr=0.1)))
    print(f'T={T:3d}  RNN  mean={np.mean(rnn_losses):.4f} std={np.std(rnn_losses):.4f}   '
          f'LSTM mean={np.mean(lstm_losses):.4f} std={np.std(lstm_losses):.4f}')

print()
print('=== 2) 단기 의존성: 시퀀스 길이별 RNN vs LSTM (대조군) ===')
for T in [5, 30]:
    ds = make_shortdep_task(10, T, seed=0)
    rnn_losses, lstm_losses = [], []
    for seed in seeds:
        rnn = SimpleRNN(1, H, seed=seed)
        rnn_losses.append(min(rnn.train(ds, epochs=150, lr=0.1)))
        lstm = SimpleLSTM(1, H, seed=seed)
        lstm_losses.append(min(lstm.train(ds, epochs=150, lr=0.1)))
    print(f'T={T:3d}  RNN  mean={np.mean(rnn_losses):.4f} std={np.std(rnn_losses):.4f}   '
          f'LSTM mean={np.mean(lstm_losses):.4f} std={np.std(lstm_losses):.4f}')

=== 1) 장기 의존성(copy task): 시퀀스 길이별 RNN vs LSTM ===
T=  5  RNN  mean=0.0008 std=0.0000   LSTM mean=0.0831 std=0.0587
T= 15  RNN  mean=0.0186 std=0.0120   LSTM mean=0.1037 std=0.0461
T= 30  RNN  mean=0.1037 std=0.0459   LSTM mean=0.1338 std=0.0034

=== 2) 단기 의존성: 시퀀스 길이별 RNN vs LSTM (대조군) ===
T=  5  RNN  mean=0.0042 std=0.0010   LSTM mean=0.0666 std=0.0024


C:\Users\ggyuh\AppData\Local\Temp\ipykernel_12432\2274873392.py:30: RuntimeWarning: overflow encountered in scalar add
  total += (target - y) ** 2
C:\Users\ggyuh\AppData\Local\Temp\ipykernel_12432\2274873392.py:30: RuntimeWarning: overflow encountered in scalar power
  total += (target - y) ** 2
C:\Users\ggyuh\AppData\Local\Temp\ipykernel_12432\2274873392.py:31: RuntimeWarning: overflow encountered in multiply
  dy = 2 * (y - target); dh = dh + dy * self.Why
C:\Users\ggyuh\AppData\Local\Temp\ipykernel_12432\2274873392.py:33: RuntimeWarning: invalid value encountered in multiply
  dt = dh * (1 - hs[t] ** 2)
C:\Users\ggyuh\AppData\Local\Temp\ipykernel_12432\2274873392.py:4: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
C:\Users\ggyuh\AppData\Local\Temp\ipykernel_12432\2274873392.py:79: RuntimeWarning: overflow encountered in scalar add
  total += (target - y) ** 2
C:\Users\ggyuh\AppData\Local\Temp\ipykernel_12432\2274873392.py:79: RuntimeWarning: overflow enc

T= 30  RNN  mean=11.1181 std=9.5074   LSTM mean=3.3957 std=0.5188


## RNN과의 비교 3: 결과 해석 및 결론

실행 결과

| 태스크 | T | RNN (mean±std) | LSTM (mean±std) |
|---|---|---|---|
| 장기 의존성 | 5  | 0.0008 ± 0.0000 | 0.0831 ± 0.0587 |
| 장기 의존성 | 15 | 0.0186 ± 0.0120 | 0.1017 ± 0.0490 |
| 장기 의존성 | 30 | 0.1037 ± 0.0459 | 0.1338 ± 0.0034 |
| 단기 의존성 | 5  | 0.0042 ± 0.0010 | 0.0666 ± 0.0024 |
| 단기 의존성 | 30 | 11.12 ± 9.51 (발산) | 3.40 ± 0.52 |

1. 이론 예측과 다른 점
- 이론상으로는 T가 커질수록 LSTM이 RNN보다 결과가 더 좋아야하는데, 실제로는 이 실험 규모에서 RNN이 평균 손실 자체는 거의 모든 조건에서 LSTM보다 낮았습니다. (더 큰 규모에서는 다를 수 있을거 같습니다.)
- 이는 LSTM의 게이트가 시그모이드 초기값(약 0.5)에서 시작해 정보를 절반쯤 걸러내며 출발하기 때문에, 학습 신호가 RNN보다 약하게 시작해서 같은 반복횟수 안에서는 덜 수렴하기 때문인거 같습니다. 
- LSTM이 항상 더 좋은결과를 보인다는 예측과는 달랐습니다.

2. LSTM이 실제로 우위를 보인 지점
- 평균 손실보다는 안정성에서 차이가 뚜렷했습니다. 
- T=30, 단기 의존성 태스크에서 학습률 0.1로 했을때 RNN에서는 기울기가 발산했지만, LSTM은 같은 조건에서 상대적으로 덜 불안정했습니다. 
- 또한 장기 의존성 T=30에서도 LSTM의 표준편차가 RNN보다 훨씬 작아, 시드를 바꿔도 결과가 일관되게 나온다는 뜻이라고 할 수 있습니다.

3. 결론
- 이번 실험 규모에서는 LSTM이 RNN보다 항상 좋은 결과를 낸다고 단정 지을 수는 없습니다. 
- 오히려 작은 데이터와 적은 반복횟수에서는 RNN이 더 빠르게 낮은 손실에 도달했습니다. 
- 그러나 시퀀스가 길어지거나 학습률이 조금만 커져도 RNN은 손실이 크게 요동치는 반면, LSTM은 게이트가 업데이트 크기를 조절해주기 때문에 훨씬 일관되고 안정적인 결과를 냈습니다. 
- 즉 LSTM의 실질적 장점은 항상 더 좋은 결과를 내주는 것이 아닌 기울기 소실 문제를 해결하여 안정적인 결과를 내준다고 할 수 있습니다. 
- 이는 이론이 예측하는 방향과도 일치하는 결과입니다.